# EX10 — Metoda Richardsona: programowe sterowanie zbieżnością

Metoda Richardsona (iteracja stacjonarna pierwszego rzędu) rozwiązuje układ $ Ax = b $ za pomocą rekurencji:

$$ x^{(k+1)} = M\,x^{(k)} + w $$

gdzie macierz iteracji oraz wektor przeniesienia mają postać

$$ M = I - p\,A, \qquad w = p\,b $$

Parametr $ p $ wpływa bezpośrednio na promień spektralny $\rho(M)$. Iteracja jest zbieżna wtedy i tylko wtedy,
gdy $\rho(M) < 1$. W ćwiczeniu sprawdzamy, jak dobór $ p $ wpływa na szybkość zbieżności,
a następnie wyprowadzamy optymalną wartość parametru.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def wilkinson(n):
    A = np.diag(np.arange(n, 0, -1), 0) + np.diag(-np.ones(n-1), 1) + np.diag(-np.ones(n-1), -1)
    return A.astype(np.float64)

A = wilkinson(5) + 10 * np.eye(5)
b = A.sum(axis=1)  # true solution is ones(5)

plt.figure(figsize=(8, 5))

for p in [0.01, 0.06]:
    M = np.eye(5) - p * A
    w_vec = p * b
    spectrumM = np.max(np.abs(np.linalg.eigvals(M)))
    print(f"p={p:.2f}, Promień spektralny: {spectrumM:.2f}")
    
    x0 = np.random.rand(5)
    x_old = x0.copy()
    err = []; n_iter = []; i = 1; error = 5e-8
    while (error >= 1e-8) and (i < 50):
        x_new = M @ x_old + w_vec
        error = np.linalg.norm(x_new - x_old)
        x_old = x_new
        err.append(error)
        n_iter.append(i)
        i += 1
    
    plt.semilogy([0] + n_iter, [5e-8] + err, label=f'p={p:.2f} (ρ={spectrumM:.2f})')

plt.legend()
plt.xlabel('# iteracji')
plt.ylabel('Estymata błędu')
plt.grid(True)
plt.show()

## ĆWICZENIE 2 — Optymalny parametr $p$ w metodzie Richardsona

*Celem jest wyprowadzenie wartości $p$ minimalizującej promień spektralny macierzy iteracji.*

---

### Podstawy teoretyczne

Rozważamy układ $Ax = b$, gdzie $A$ jest macierzą symetryczną dodatnio określoną.
Wartości własne $A$ są rzeczywiste i dodatnie:

$$ \lambda_1 \le \lambda_2 \le \dots \le \lambda_n $$

Wartości własne macierzy iteracji $M = I - p\,A$ wynoszą:

$$ \mu_i(M) = 1 - p\,\lambda_i \qquad (i=1,\dots,n) $$

Warunek zbieżności iteracji wymaga:

$$ \rho(M) = \max_{i} |1 - p\,\lambda_i| < 1 $$

co implikuje ograniczenie na parametr:

$$ 0 < p < \frac{2}{\lambda_{\max}} $$

---

### Wyprowadzenie optymalnego $p$

Szukamy $p$ minimalizującego $\rho(M)$. Ponieważ wartości własne $\lambda_i > 0$, promień spektralny to:

$$ \rho(M) = \max\bigl(\,|1 - p\,\lambda_{\min}|,\; |1 - p\,\lambda_{\max}|\,\bigr) $$

Minimum występuje, gdy oba wyrazy są równe co do wartości bezwzględnej:

$$ 1 - p\,\lambda_{\min} = -(1 - p\,\lambda_{\max}) $$

Stąd:

$$ p_{\mathrm{opt}} = \frac{2}{\lambda_{\min} + \lambda_{\max}} $$

---

### Promień spektralny w punkcie optymalnym

$$ \rho(M_{\mathrm{opt}}) = \frac{\lambda_{\max} - \lambda_{\min}}{\lambda_{\max} + \lambda_{\min}} = \frac{\kappa(A) - 1}{\kappa(A) + 1} $$

gdzie $\kappa(A) = \lambda_{\max}/\lambda_{\min}$ to liczba warunkowa macierzy $A$ w normie spektralnej.

---

### Wnioski

1. Im mniejsza liczba warunkowa $\kappa(A)$, tym szybsza zbieżność.
2. Dla $\kappa(A) \gg 1$ promień spektralny zbliża się do $1$ i iteracja staje się bardzo powolna.
3. Parametr $\beta$ w notacji $M = I - \beta A$ odpowiada $p$ w naszym zapisie.

In [ ]:
# Weryfikacja teoretycznych wzorów

eigenvalues = np.linalg.eigvals(A)
lam_min = np.min(np.real(eigenvalues))
lam_max = np.max(np.real(eigenvalues))
kappa = lam_max / lam_min

p_opt = 2 / (lam_min + lam_max)
rho_opt = (kappa - 1) / (kappa + 1)

print(f"λ_min = {lam_min:.4f}")
print(f"λ_max = {lam_max:.4f}")
print(f"κ(A)   = {kappa:.4f}")
print(f"p_opt   = {p_opt:.6f}")
print(f"ρ(M_opt) (teoria) = {rho_opt:.6f}")

# Weryfikacja numeryczna
M_opt = np.eye(5) - p_opt * A
rho_num = np.max(np.abs(np.linalg.eigvals(M_opt)))
print(f"ρ(M_opt) (numerycznie) = {rho_num:.6f}")

# Porównanie z p = 0.06
rho_006 = np.max(np.abs(np.linalg.eigvals(np.eye(5) - 0.06 * A)))
print(f"ρ dla p=0.06 = {rho_006:.6f}")

## Wykorzystywane funkcje

| Funkcja | Opis |
| --- | --- |
| `np.linalg.eigvals(A)` | Oblicza wartości własne macierzy $A$ |
| `wilkinson(n)` | Generuje macierz Wilkinsona rozmiaru $n\times n$ |
| `np.linalg.norm(v)` | Oblicza normę (długość) wektora $v$ |
| `np.eye(n)` | Tworzy macierz jednostkową $n\times n$ |
| `np.diag(d, k)` | Tworzy macierz diagonalną z wektora $d$ na $k$-tej diagonali |
| `np.random.rand(n)` | Generuje wektor $n$-wymiarowy o losowych składnikach z $U(0,1)$ |
| `plt.semilogy(x, y)` | Wykres liniowy z logarytmiczną osią $y$ |
| `A.sum(axis=1)` | Suma wierszy macierzy $A$ (powrót jako wektor kolumnowy) |